In [2]:
# -----------------------------
# 1. Imports
# -----------------------------
import os
import numpy as np
import librosa
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset

# -----------------------------
# 2. Dataset Path
# -----------------------------
DATASET_PATH = r"C:\Users\LENOVO\Desktop\deep fake dataset\voice dataset"

# -----------------------------
# 3. Gather audio files and labels
# -----------------------------
labels_list = ["real", "fake"]
label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for i, label in enumerate(labels_list)}

file_paths, file_labels = [], []

for label in labels_list:
    label_path = os.path.join(DATASET_PATH, label)
    if not os.path.exists(label_path):
        print(f"⚠️ Warning: Folder not found: {label_path}")
        continue

    for lang_folder in os.listdir(label_path):
        lang_path = os.path.join(label_path, lang_folder)
        if not os.path.isdir(lang_path):
            continue

        for file in os.listdir(lang_path):
            file_path = os.path.join(lang_path, file)
            if file.lower().endswith(".wav"):
                file_paths.append(file_path)
                file_labels.append(label2id[label])

if len(file_paths) == 0:
    raise ValueError("❌ No audio files found! Check dataset path and folder names.")

# -----------------------------
# 4. Shuffle and Split (80/20)
# -----------------------------
combined = list(zip(file_paths, file_labels))
np.random.shuffle(combined)
file_paths, file_labels = zip(*combined)

train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, file_labels, test_size=0.2, stratify=file_labels, random_state=42
)

print(f"✅ Total files: {len(file_paths)}")
print(f"🟢 Train: {len(train_paths)} | 🔵 Test: {len(test_paths)}")

# -----------------------------
# 5. Load model and processor
# -----------------------------
model_name = "facebook/wav2vec2-base"
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    label2id=label2id,
    id2label=id2label,
    use_safetensors=True
)
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

# -----------------------------
# 6. Audio preprocessing
# -----------------------------
MAX_DURATION = 10  # seconds

def preprocess_audio(file_path, label):
    try:
        speech, sr = librosa.load(file_path, sr=16000)
        speech = librosa.util.normalize(speech)

        max_len = MAX_DURATION * sr
        if len(speech) > max_len:
            speech = speech[:max_len]
        else:
            speech = np.pad(speech, (0, max_len - len(speech)), "constant")

        inputs = processor(
            speech,
            sampling_rate=sr,
            return_tensors="pt",
            padding=True
        )

        return {"input_values": inputs.input_values[0], "labels": label}
    except Exception as e:
        print(f"⚠️ Error processing {file_path}: {e}")
        return None

# -----------------------------
# 7. Create Hugging Face Datasets
# -----------------------------
def dataset_generator(paths, labels):
    for p, l in zip(paths, labels):
        data = preprocess_audio(p, l)
        if data:
            yield data

train_dataset = Dataset.from_generator(lambda: dataset_generator(train_paths, train_labels))
test_dataset = Dataset.from_generator(lambda: dataset_generator(test_paths, test_labels))

print(f"✅ Dataset ready: {len(train_dataset)} train, {len(test_dataset)} test samples")

# -----------------------------
# 8. Metrics
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# -----------------------------
# 9. Data collator
# -----------------------------
data_collator = DataCollatorWithPadding(processor, padding=True)

# -----------------------------
# 10. Training configuration
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    fp16=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    logging_dir="./logs"
)

# -----------------------------
# 11. Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# -----------------------------
# 12. Train
# -----------------------------
trainer.train()

# -----------------------------
# 13. Save best model
# -----------------------------
save_dir = "./wav2vec2_base_ai_detector_v2"
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

print(f"✅ Training complete. Model saved at: {save_dir}")


C:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Total files: 3559
🟢 Train: 2847 | 🔵 Test: 712


C:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Generating train split: 2847 examples [00:44, 64.66 examples/s] 
Generating train split: 712 examples [00:12, 55.76 examples/s] 
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12408\1128114515.py:157: FutureWarning: `tokenizer` is deprecated and

✅ Dataset ready: 2847 train, 712 test samples


Step,Training Loss
10,0.655500
20,0.576200
30,0.495300
40,0.351400
50,0.367600
60,0.316100
70,0.208800
80,0.291700
90,0.359500
100,0.244300


✅ Training complete. Model saved at: ./wav2vec2_base_ai_detector_v2


In [1]:
import torch
torch.cuda.empty_cache()


In [3]:
import torch
print(torch.__version__)


2.6.0.dev20241112+cu121


In [2]:
import shutil
import os

cache_dir = os.path.expanduser("~/.cache/huggingface/transformers")
shutil.rmtree(cache_dir, ignore_errors=True)


In [1]:
import torch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [4]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Define a compute_metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Re-run evaluation on test set
results = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
print("📊 Evaluation Results:", results)


📊 Evaluation Results: {'test_loss': 0.025929413735866547, 'test_accuracy': 0.9971910112359551, 'test_f1': 0.9927007299270073, 'test_runtime': 211.6101, 'test_samples_per_second': 3.365, 'test_steps_per_second': 3.365, 'epoch': 10.0}


In [6]:
import torch
import librosa
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor,Wav2Vec2FeatureExtractor

# -----------------------------
# 1. Load model & processor
# -----------------------------
model_path = "./wav2vec2_base_ai_detector_v2"   # <-- update with your model folder name
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

# Load processor (handles feature extraction)
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_path)

# -----------------------------
# 2. Prediction function
# -----------------------------
def predict_audio(file_path):
    # Load and resample audio to 16kHz
    speech, sr = librosa.load(file_path, sr=16000)

    # Convert to tensor input
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)

    # Run through model
    with torch.no_grad():
        outputs = model(input_values)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=-1).cpu().item()

    # -------------------------
    # Label mapping (adjust if flipped)
    # -------------------------
    label_map = {0: "Real", 1: "AI-Generated"}
    return label_map[predicted_class]

# -----------------------------
# 3. Test on an audio sample
# -----------------------------
test_file = r"C:\Users\LENOVO\Desktop\WhatsApp Ptt 2025-10-04 at 9.28.26 PM.ogg"

result = predict_audio(test_file)
print(f"🎧 Prediction for {test_file.split('\\')[-1]}: {result}")


🎧 Prediction for WhatsApp Ptt 2025-10-04 at 9.28.26 PM.ogg: Real
